# Task 2: Real Estate Housing Price Prediction (OLS, Ridge & Lasso)
**Goal:** Predict median house values on the California Housing dataset, engineer interaction features, impute missing values, handle multicollinearity, and compare OLS, Ridge (L2) and Lasso (L1) using RMSE, MAE and R².

**Data:** place the starter file at `data/housing.csv` (columns: longitude, latitude, housing_median_age, total_rooms, total_bedrooms, population, households, median_income, median_house_value, ocean_proximity). Adjust `DATA_PATH` if your file is named differently.

## 1. Imports and data

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

sns.set_style("whitegrid")
RANDOM_STATE = 42
DATA_PATH = Path("../data/housing.csv")
TARGET = "median_house_value"

df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
display(df.head())
df.info()
print("Missing values per column:")
print(df.isna().sum())

## 2. Exploratory analysis: target distribution and correlation heatmap

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(df[TARGET], bins=50, kde=True, ax=axes[0])
axes[0].set_title("Distribution of median house value")
sns.heatmap(df.select_dtypes("number").corr(), annot=True, fmt=".2f",
            cmap="coolwarm", center=0, ax=axes[1], annot_kws={"size": 8})
axes[1].set_title("Correlation heatmap (raw numeric features)")
plt.tight_layout()
plt.show()

## 3. Feature engineering
Ratio and interaction features describe each block more meaningfully than raw totals. Missing values created here (for example from a missing `total_bedrooms`) are imputed later inside the pipeline, so imputation statistics come from the training data only.

In [ ]:
df["rooms_per_household"] = df["total_rooms"] / df["households"]
df["bedrooms_per_room"] = df["total_bedrooms"] / df["total_rooms"]
df["population_per_household"] = df["population"] / df["households"]
df["income_x_age"] = df["median_income"] * df["housing_median_age"]   # interaction feature
df["income_x_rooms"] = df["median_income"] * df["rooms_per_household"]  # interaction feature
display(df[["rooms_per_household", "bedrooms_per_room", "population_per_household",
            "income_x_age", "income_x_rooms"]].describe().round(2))

## 4. Multicollinearity check (Variance Inflation Factor)
VIF above about 10 signals a feature that is largely explained by the others. The raw count columns (`total_rooms`, `total_bedrooms`, `population`, `households`) are strongly related to each other.

In [ ]:
def vif_table(frame):
    X = pd.DataFrame(SimpleImputer(strategy="median").fit_transform(frame), columns=frame.columns)
    X = (X - X.mean()) / X.std()
    vif = {}
    for c in X.columns:
        others = X.drop(columns=c)
        r2 = LinearRegression().fit(others, X[c]).score(others, X[c])
        vif[c] = 1 / (1 - r2) if r2 < 1 else np.inf
    return pd.Series(vif, name="VIF").sort_values(ascending=False)

num_features = df.select_dtypes("number").drop(columns=TARGET)
print("VIF before handling multicollinearity:")
display(vif_table(num_features).round(1).to_frame())

# The raw totals are replaced by the per-household ratios, so drop them.
DROP = ["total_rooms", "total_bedrooms", "population"]
print("VIF after dropping", DROP)
display(vif_table(num_features.drop(columns=DROP)).round(1).to_frame())

Some VIF remains (for example between latitude and longitude, and the interaction terms, which are built from other features). Ridge and Lasso are designed to cope with this, which is part of the comparison below.

## 5. Train/test split and preprocessing pipeline
Median imputation and `StandardScaler` for numeric columns, `OneHotEncoder` for categorical ones. Everything is fit on the training data only.

In [ ]:
X = df.drop(columns=[TARGET] + DROP)
y = df[TARGET]
cat_cols = X.select_dtypes(exclude="number").columns.tolist()
num_cols = X.select_dtypes("number").columns.tolist()
print("Numeric:", num_cols)
print("Categorical:", cat_cols)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=RANDOM_STATE)

def make_pre():
    transformers = [("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                                      ("sc", StandardScaler())]), num_cols)]
    if cat_cols:
        transformers.append(("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                                              ("oh", OneHotEncoder(handle_unknown="ignore"))]), cat_cols))
    return ColumnTransformer(transformers, verbose_feature_names_out=False)

## 6. Fit OLS, and tune Ridge and Lasso with GridSearchCV (5-fold CV)

In [ ]:
ols = Pipeline([("pre", make_pre()), ("model", LinearRegression())]).fit(X_train, y_train)

ridge_gs = GridSearchCV(
    Pipeline([("pre", make_pre()), ("model", Ridge())]),
    {"model__alpha": np.logspace(-3, 3, 13)},
    cv=5, scoring="neg_root_mean_squared_error", n_jobs=-1).fit(X_train, y_train)

lasso_gs = GridSearchCV(
    Pipeline([("pre", make_pre()), ("model", Lasso(max_iter=50000))]),
    {"model__alpha": np.logspace(-3, 2, 11)},
    cv=5, scoring="neg_root_mean_squared_error", n_jobs=-1).fit(X_train, y_train)

print("Best Ridge alpha:", ridge_gs.best_params_["model__alpha"])
print("Best Lasso alpha:", lasso_gs.best_params_["model__alpha"])

models = {"OLS": ols, "Ridge (L2)": ridge_gs.best_estimator_, "Lasso (L1)": lasso_gs.best_estimator_}

## 7. Model comparison: RMSE, MAE, R²

In [ ]:
rows, preds = [], {}
for name, m in models.items():
    p = m.predict(X_test)
    preds[name] = p
    rows.append({"Model": name,
                 "RMSE": np.sqrt(mean_squared_error(y_test, p)),
                 "MAE": mean_absolute_error(y_test, p),
                 "R²": r2_score(y_test, p)})
results = pd.DataFrame(rows).set_index("Model").round(4)
display(results)

## 8. Predicted vs actual

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharex=True, sharey=True)
lims = [min(y_test.min(), min(p.min() for p in preds.values())),
        max(y_test.max(), max(p.max() for p in preds.values()))]
for ax, (name, p) in zip(axes, preds.items()):
    ax.scatter(y_test, p, alpha=0.3, s=10)
    ax.plot(lims, lims, "r--", lw=1.5, label="Perfect prediction")
    ax.set_title(f"{name}  (R² = {r2_score(y_test, p):.3f})")
    ax.set_xlabel("Actual value")
    ax.legend()
axes[0].set_ylabel("Predicted value")
plt.tight_layout()
plt.show()

## 9. Residual distributions

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for i, (name, p) in enumerate(preds.items()):
    resid = y_test - p
    sns.histplot(resid, bins=50, kde=True, ax=axes[0, i])
    axes[0, i].axvline(0, color="r", ls="--")
    axes[0, i].set_title(f"{name}: residual distribution")
    axes[1, i].scatter(p, resid, alpha=0.3, s=10)
    axes[1, i].axhline(0, color="r", ls="--")
    axes[1, i].set_title(f"{name}: residuals vs predicted")
    axes[1, i].set_xlabel("Predicted value")
    axes[1, i].set_ylabel("Residual")
plt.tight_layout()
plt.show()

## 10. Coefficients: how regularization changes the model

In [ ]:
names = ols.named_steps["pre"].get_feature_names_out()
coefs = pd.DataFrame({name: m.named_steps["model"].coef_ for name, m in models.items()}, index=names)
top = coefs["OLS"].abs().sort_values(ascending=False).head(12).index
coefs.loc[top].plot(kind="barh", figsize=(9, 7), title="Top coefficients (standardized features)")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

n_zero = int((coefs["Lasso (L1)"].abs() < 1e-8).sum())
print(f"Lasso set {n_zero} of {len(coefs)} coefficients exactly to zero.")

## 11. Conclusions
- **OLS** is the unregularized baseline. With few, mostly independent features its error is similar to the regularized models.
- **Ridge (L2)** shrinks correlated coefficients towards each other and stabilizes them without removing any feature.
- **Lasso (L1)** can set weak coefficients exactly to zero, acting as automatic feature selection.
- The residual plots show whether errors are centred on zero and whether the model struggles at high prices (the dataset caps house values, which produces a visible band at the top).
- Compare the table in section 7 and write your own interpretation of which model performs best and why.